# IR System — Evaluation

Metrics: **MAP, Recall, Precision@10, nDCG** for every representation, over **all** queries in the qrels file.

> The number of queries used is printed below (computed live from the qrels file).

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from irsys import config
from irsys.data import loaders

DATASET = 'quora'
qrels = loaders.load_qrels(DATASET)
queries = loaders.load_queries(DATASET)
eval_qids = [q for q in qrels if q in queries and any(r > 0 for r in qrels[q].values())]
print('Dataset           :', config.DATASETS[DATASET]['label'], '(', config.DATASETS[DATASET]['irds_id'], ')')
print('NUMBER OF QUERIES USED =', len(eval_qids))

## Results
Loads the metrics produced by `python scripts/evaluate.py --dataset quora --tag baseline`.
(Run that script first; it evaluates all methods over all queries.)

In [ ]:
import pandas as pd
tag = 'baseline'
rep = json.loads((config.REPORTS_DIR / f'eval_{DATASET}_{tag}.json').read_text(encoding='utf-8'))
print('queries in report:', rep['num_queries'])
df = pd.DataFrame(rep['results']).T
df

In [ ]:
import matplotlib.pyplot as plt
metric_cols = [c for c in df.columns if c != 'num_queries']
df[metric_cols].plot(kind='bar', figsize=(11,6))
plt.title(f'IR evaluation — {DATASET} ({tag})'); plt.ylabel('score'); plt.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

## (Optional) Recompute one method live
Demonstrates the evaluation is genuine — recomputes BM25 over all queries against the open qrels file.

In [ ]:
from irsys.pipeline import RetrievalEngine
from irsys.evaluation import evaluate_run
eng = RetrievalEngine.load(DATASET)
run = {qid: [d for d,_ in eng.search('bm25', queries[qid], top_k=100)] for qid in eval_qids}
ev = evaluate_run(run, qrels)
print('queries used:', ev['metrics']['num_queries'])
print({k: round(v,4) for k,v in ev['metrics'].items() if k!='num_queries'})

## Before vs after extra features
Compare the `baseline` report with the `with_features` report (clustering, etc.).

In [ ]:
p2 = config.REPORTS_DIR / f'eval_{DATASET}_with_features.json'
if p2.exists():
    rep2 = json.loads(p2.read_text(encoding='utf-8'))
    print('with_features queries:', rep2['num_queries'])
    display(pd.DataFrame(rep2['results']).T)
else:
    print('Run: python scripts/evaluate.py --dataset quora --methods embedding --tag with_features  (after building features)')